# 01｜第 1 節：讓 AI 猜小考分數

**讀資料 → 分組 → 訓練直線 → 看 MSE／R²**。第 1 格已寫好匯入模組與讀取資料，直接執行；第 2、3 格依 Markdown 範例輸入到空白儲存格；最後兩格 Optional 可直接執行。CSV 請與 Notebook 放在同一資料夾。

### 資料集欄位（500 筆）

| 欄位 | 意義 | 本次用途 |
|---|---|---|
| `study_hours` | 每天讀書幾小時，有空白與一筆 60 小時 | 第 1 節的輸入 X |
| `quiz_score` | 小考分數（0～100） | 第 1 節答案 y |
| `practice_questions` | 每天練習幾題，有空白 | 第 2 節的輸入 X |
| `passed` | 是否及格：0＝未及格、1＝及格 | 第 2 節答案 y |
| 其餘欄位 | 學號、學習方式、出席率、睡眠、努力程度 | 本次不用 |

`X` 是模型看到的資料，`y` 是要猜的答案。不要把 `quiz_score` 或 `passed` 偷放到同一個任務的 `X`。

### 步驟 1/3｜匯入模組與讀取資料（直接執行）

找到輸入 `study_hours` 與答案 `quiz_score`；不合理的 60 小時先改成空白。

**這一格已經寫好，直接按 Shift + Enter 執行，不需要重新輸入。**

In [ ]:
# Import NumPy for marking invalid entries as missing.
# 匯入 NumPy，將錯誤數值標記為空白。
import numpy as np

# Import pandas to load the dataset.
# 匯入 pandas，讀取資料表。
import pandas as pd

# Import matplotlib for the optional graph.
# 匯入畫圖套件，供選做實驗使用。
import matplotlib.pyplot as plt

# Import the training/test splitting function.
# 匯入訓練集與測試集切分工具。
from sklearn.model_selection import train_test_split

# Import the linear regression model.
# 匯入線性迴歸模型。
from sklearn.linear_model import LinearRegression

# Import the two regression metrics.
# 匯入 MSE 與 R² 兩個評估工具。
from sklearn.metrics import mean_squared_error, r2_score

# Load the provided 500-row CSV dataset.
# 載入提供的 500 筆練習資料。
df = pd.read_csv('01_機器學習_學生練習資料_500筆.csv')

# Mark the impossible 60-hour record as missing.
# 將一天 60 小時的不合理紀錄標成空白。
df.loc[df['study_hours'] > 24, 'study_hours'] = np.nan

# Preview only the input and target columns.
# 預覽讀書時數與小考分數兩欄。
print(df[['study_hours', 'quiz_score']].head())

### 必做 2/3｜留下測試題

把 500 筆分成 80% 訓練、20% 測試。空白時數使用訓練組的中位數補齊。

```python
# Select the study-hours input column.
# 選擇讀書時數作為模型輸入 X。
X = df[['study_hours']]

# Select the score column to predict.
# 選擇小考分數作為答案 y。
y = df['quiz_score']

# Keep one fifth of rows for unseen testing.
# 保留五分之一的資料作為測試集。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Learn the median using training data only.
# 只用訓練資料算出補空白用的中位數。
median_hours = X_train['study_hours'].median()

# Fill missing hours in the training set.
# 用訓練組中位數補齊訓練組空白。
X_train = X_train.fillna(median_hours)

# Use the same median for unseen test rows.
# 測試組也使用相同中位數補空白。
X_test = X_test.fillna(median_hours)

# Check how many rows are in each group.
# 確認訓練組與測試組各有幾筆。
print('訓練：', len(X_train), '筆；測試：', len(X_test), '筆')
```

In [ ]:
# 將上方「必做 2/3」範例輸入此格，再按 Shift + Enter。

### 必做 3/3｜訓練迴歸並評分

用 `LinearRegression()` 學習一條直線，只看 **MSE（越小越好）** 與 **R²（1 最好；0 等於猜平均值）**。

```python
# Create a model that learns a straight line.
# 建立會學習一條直線的迴歸模型。
model = LinearRegression()

# Fit the model on training examples.
# 用訓練組資料學習。
model.fit(X_train, y_train)

# Predict scores for unseen test examples.
# 預測沒看過的測試組分數。
pred = model.predict(X_test)

# Measure the average squared prediction error.
# 計算平均平方誤差 MSE。
mse = mean_squared_error(y_test, pred)

# Compare the model with guessing the mean.
# 計算與只猜平均數相比的 R²。
r2 = r2_score(y_test, pred)

# Display only the two core evaluation metrics.
# 只顯示本課需要的兩個評估數字。
print(f'MSE = {mse:.2f}，R² = {r2:.2f}')
```

In [ ]:
# 將上方「必做 3/3」範例輸入此格，再按 Shift + Enter。

In [ ]:
# Optional experiment: 看資料點和預測直線
# 選做實驗：看資料點和預測直線
# Draw the training examples as data points.
# 畫出訓練資料的實際散佈點。
plt.scatter(X_train['study_hours'], y_train, alpha=0.35, label='實際分數')

# Create six increasing study-hour values.
# 建立六個由小到大的讀書時數。
line_x = pd.DataFrame({'study_hours': [0, 2, 4, 6, 8, 10]})

# Draw the model predictions as a line.
# 把模型預測結果畫成直線。
plt.plot(line_x['study_hours'], model.predict(line_x), color='orange', linewidth=3, label='模型預測')

# Name the horizontal axis.
# 標示橫軸為讀書時數。
plt.xlabel('Study hours')

# Name the vertical axis.
# 標示縱軸為測驗分數。
plt.ylabel('Quiz score')

# Show the chart legend.
# 顯示圖例。
plt.legend()

# Display the completed regression graph.
# 顯示完成的迴歸圖表。
plt.show()


In [ ]:
# Optional experiment: 輸入三筆新資料預測分數
# 選做實驗：輸入三筆新資料預測分數
# Create three unseen study-hour examples.
# 建立三筆新的讀書時數範例。
new_data = pd.DataFrame({'study_hours': [1, 3, 5]})

# Predict each example without fitting again.
# 直接預測，不重新訓練模型。
new_data['預測分數'] = model.predict(new_data).round(1)

# Display the three predictions.
# 查看三筆預測結果。
print(new_data)
